# Vẽ đồ thị so sánh thực nghiệm

Notebook này đọc trực tiếp các file kết quả trong `results/` và vẽ đồ thị cho phần so sánh thực nghiệm:

- Thay đổi ngân sách chặn `k = 100, 200, 300, 400, 500` khi cố định `|S| = 10`.
- Thay đổi số seed tin đồn `|S| = 10, 20, 30, 40, 50` khi cố định `k = 100`.
- So sánh các biến thể `BEST`, `LB`, `UB`, `OR` của SandIMIN.

Notebook chỉ cần thư viện chuẩn Python và `matplotlib`, không cần `pandas`.

In [2]:
from pathlib import Path
from collections import defaultdict
import csv
import math
import os
import statistics

# Tránh lỗi quyền ghi cache của Matplotlib trong một số môi trường notebook/sandbox.
mpl_cache = Path.cwd() / ".matplotlib_cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache.resolve()))
os.environ.setdefault("XDG_CACHE_HOME", str(mpl_cache.resolve()))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (12, 7),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

## 1. Cấu hình đường dẫn

In [3]:
def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "results").exists() and (path / "docs").exists():
            return path
    raise FileNotFoundError("Không tìm thấy project root có thư mục results/ và docs/.")


ROOT = find_project_root()
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

ROOT: /Users/apple/Documents/Ki_2_nam_3/Cơ sở ATTT/finalBTL/ATBM
RESULTS_DIR: /Users/apple/Documents/Ki_2_nam_3/Cơ sở ATTT/finalBTL/ATBM/results


## 2. Đọc dữ liệu kết quả

IMin/SandIMIN ghi các file `res_*.txt` dạng TSV. Nếu sau này có `ag_gr_results.txt` hoặc `ag_results.tsv`, notebook cũng đọc thêm để so sánh với AG/GR.

In [4]:
def parse_float(value):
    if value in (None, "", "-", "NA"):
        return math.nan
    try:
        return float(value)
    except ValueError:
        return math.nan


def parse_int(value):
    return int(float(value))


def read_sandimin_results(results_dir):
    rows = []
    for path in sorted(results_dir.glob("res_*.txt")):
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            reader = csv.DictReader(f, delimiter="\t")
            if not reader.fieldnames or "algorithm" not in reader.fieldnames:
                continue
            for row in reader:
                algorithm = row.get("algorithm", "SandIMIN")
                variant = row.get("result_type", "") or "RESULT"
                rows.append({
                    "algorithm": algorithm,
                    "variant": variant,
                    "method": f"{algorithm}-{variant}",
                    "dataset": row.get("dataset", ""),
                    "k": parse_int(row.get("k", 0)),
                    "seed_num": parse_int(row.get("|S|", 0)),
                    "spread_before": parse_float(row.get("spread_before")),
                    "spread_after": parse_float(row.get("spread_after")),
                    "saved_nodes": parse_float(row.get("saved_nodes")),
                    "time_seconds": parse_float(row.get("time_seconds")),
                    "approx_ratio": parse_float(row.get("approx_ratio")),
                    "status": row.get("status", "OK"),
                    "source_file": path.name,
                })
    return rows


def read_ag_results(results_dir):
    rows = []
    for path in [results_dir / "ag_gr_results.txt", results_dir / "ag_results.tsv"]:
        if not path.exists():
            continue
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or line.startswith("=") or line.startswith("Format") or line == "---":
                    continue
                parts = line.split("\t")
                if len(parts) < 9:
                    continue
                algorithm, dataset, k, seed_num = parts[:4]
                rows.append({
                    "algorithm": algorithm,
                    "variant": "RESULT",
                    "method": algorithm,
                    "dataset": dataset,
                    "k": parse_int(k),
                    "seed_num": parse_int(seed_num),
                    "spread_before": math.nan,
                    "spread_after": math.nan,
                    "saved_nodes": parse_float(parts[6]),
                    "time_seconds": parse_float(parts[7]),
                    "approx_ratio": math.nan,
                    "status": parts[8],
                    "source_file": path.name,
                })
    return rows


raw = read_sandimin_results(RESULTS_DIR) + read_ag_results(RESULTS_DIR)
if not raw:
    raise FileNotFoundError(f"Không có dữ liệu kết quả trong {RESULTS_DIR}")

print("Số dòng raw:", len(raw))
print("Dataset:", ", ".join(sorted({r["dataset"] for r in raw})))
print("Method:", ", ".join(sorted({r["method"] for r in raw})))
raw[:3]

Số dòng raw: 418
Dataset: com-dblp.ungraph, com-youtube.ungraph, email-EuAll, p2p-Gnutella31
Method: SandIMIN-BEST, SandIMIN-LB, SandIMIN-OR, SandIMIN-UB


[{'algorithm': 'SandIMIN',
  'variant': 'BEST',
  'method': 'SandIMIN-BEST',
  'dataset': 'com-dblp.ungraph',
  'k': 100,
  'seed_num': 10,
  'spread_before': 709.23,
  'spread_after': 506.893,
  'saved_nodes': 202.338,
  'time_seconds': 6389.8,
  'approx_ratio': nan,
  'status': 'OK',
  'source_file': 'res_com-dblp.ungraph_S=10_K=100_epsilon=0.200000_gamma=0.100000_beta=0.100000_algo=SandIMIN.txt'},
 {'algorithm': 'SandIMIN',
  'variant': 'LB',
  'method': 'SandIMIN-LB',
  'dataset': 'com-dblp.ungraph',
  'k': 100,
  'seed_num': 10,
  'spread_before': 709.23,
  'spread_after': 506.622,
  'saved_nodes': 202.608,
  'time_seconds': 490.864,
  'approx_ratio': nan,
  'status': 'OK',
  'source_file': 'res_com-dblp.ungraph_S=10_K=100_epsilon=0.200000_gamma=0.100000_beta=0.100000_algo=SandIMIN.txt'},
 {'algorithm': 'SandIMIN',
  'variant': 'UB',
  'method': 'SandIMIN-UB',
  'dataset': 'com-dblp.ungraph',
  'k': 100,
  'seed_num': 10,
  'spread_before': 709.23,
  'spread_after': 531.963,
  'sa

## 3. Tổng hợp trung bình và độ lệch chuẩn

In [5]:
def mean(values):
    values = [v for v in values if not math.isnan(v)]
    return statistics.mean(values) if values else math.nan


def stdev(values):
    values = [v for v in values if not math.isnan(v)]
    return statistics.stdev(values) if len(values) >= 2 else 0.0


grouped = defaultdict(list)
for row in raw:
    key = (row["dataset"], row["method"], row["variant"], row["k"], row["seed_num"])
    grouped[key].append(row)

summary = []
for (dataset, method, variant, k, seed_num), rows in grouped.items():
    summary.append({
        "dataset": dataset,
        "method": method,
        "variant": variant,
        "k": k,
        "seed_num": seed_num,
        "runs": len(rows),
        "saved_nodes_mean": mean([r["saved_nodes"] for r in rows]),
        "saved_nodes_std": stdev([r["saved_nodes"] for r in rows]),
        "time_seconds_mean": mean([r["time_seconds"] for r in rows]),
        "time_seconds_std": stdev([r["time_seconds"] for r in rows]),
        "approx_ratio_mean": mean([r["approx_ratio"] for r in rows]),
        "approx_ratio_std": stdev([r["approx_ratio"] for r in rows]),
    })

summary = sorted(summary, key=lambda r: (r["dataset"], r["method"], r["k"], r["seed_num"]))
summary[:10]

[{'dataset': 'com-dblp.ungraph',
  'method': 'SandIMIN-BEST',
  'variant': 'BEST',
  'k': 100,
  'seed_num': 10,
  'runs': 3,
  'saved_nodes_mean': 205.00233333333333,
  'saved_nodes_std': 2.37029752000321,
  'time_seconds_mean': 4525.57,
  'time_seconds_std': 1617.236493744808,
  'approx_ratio_mean': nan,
  'approx_ratio_std': 0.0},
 {'dataset': 'com-dblp.ungraph',
  'method': 'SandIMIN-BEST',
  'variant': 'BEST',
  'k': 100,
  'seed_num': 20,
  'runs': 3,
  'saved_nodes_mean': 284.586,
  'saved_nodes_std': 3.1646122985288403,
  'time_seconds_mean': 4621.83,
  'time_seconds_std': 1661.1457849327974,
  'approx_ratio_mean': nan,
  'approx_ratio_std': 0.0},
 {'dataset': 'com-dblp.ungraph',
  'method': 'SandIMIN-BEST',
  'variant': 'BEST',
  'k': 100,
  'seed_num': 30,
  'runs': 2,
  'saved_nodes_mean': 334.40200000000004,
  'saved_nodes_std': 6.505382386916229,
  'time_seconds_mean': 2109.435,
  'time_seconds_std': 311.91187224919776,
  'approx_ratio_mean': nan,
  'approx_ratio_std': 0.0

## 4. Hàm vẽ đồ thị

Mỗi dataset là một subplot. Mỗi đường là một `method`: `SandIMIN-BEST`, `SandIMIN-LB`, `SandIMIN-UB`, `SandIMIN-OR`.

In [6]:
METHOD_ORDER = ["SandIMIN-BEST", "SandIMIN-LB", "SandIMIN-UB", "SandIMIN-OR", "AG", "GR"]
METHOD_COLORS = {
    "SandIMIN-BEST": "#1f77b4",
    "SandIMIN-LB": "#2ca02c",
    "SandIMIN-UB": "#ff7f0e",
    "SandIMIN-OR": "#d62728",
    "AG": "#9467bd",
    "GR": "#8c564b",
}
METHOD_MARKERS = {
    "SandIMIN-BEST": "o",
    "SandIMIN-LB": "s",
    "SandIMIN-UB": "^",
    "SandIMIN-OR": "D",
    "AG": "P",
    "GR": "X",
}


def ordered_methods(rows):
    present = {r["method"] for r in rows}
    ordered = [m for m in METHOD_ORDER if m in present]
    ordered += sorted(m for m in present if m not in ordered)
    return ordered


def filter_rows(rows, filters):
    out = []
    for row in rows:
        ok = True
        for col, value in filters.items():
            if row[col] != value:
                ok = False
                break
        if ok:
            out.append(row)
    return out


def plot_metric_by_dataset(rows, x_col, metric_prefix, fixed_filters, title, xlabel, ylabel, filename=None, yscale="linear"):
    mean_col = f"{metric_prefix}_mean"
    std_col = f"{metric_prefix}_std"
    data = [r for r in filter_rows(rows, fixed_filters) if not math.isnan(r[mean_col])]
    if not data:
        print(f"Không có dữ liệu cho {title}")
        return None

    datasets = sorted({r["dataset"] for r in data})
    ncols = 2
    nrows = math.ceil(len(datasets) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, dataset in zip(axes_flat, datasets):
        sub = [r for r in data if r["dataset"] == dataset]
        for method in ordered_methods(sub):
            line = sorted([r for r in sub if r["method"] == method], key=lambda r: r[x_col])
            if not line:
                continue
            ax.errorbar(
                [r[x_col] for r in line],
                [r[mean_col] for r in line],
                yerr=[r[std_col] for r in line],
                marker=METHOD_MARKERS.get(method, "o"),
                linewidth=2,
                capsize=3,
                label=method,
                color=METHOD_COLORS.get(method),
            )
        ax.set_title(dataset)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_yscale(yscale)
        ax.legend(fontsize=9)

    for ax in axes_flat[len(datasets):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=15, y=1.02)
    fig.tight_layout()

    if filename:
        out = FIGURES_DIR / filename
        fig.savefig(out, bbox_inches="tight")
        print("Đã lưu:", out)

    return fig

## 5. Kịch bản 1: thay đổi `k`, cố định `|S| = 10`

In [7]:
plot_metric_by_dataset(
    summary,
    x_col="k",
    metric_prefix="saved_nodes",
    fixed_filters={"seed_num": 10},
    title="Số nút cứu được khi thay đổi k, cố định |S| = 10",
    xlabel="k - số node được chặn",
    ylabel="Số nút cứu được trung bình",
    filename="saved_nodes_by_k.png",
);

Đã lưu: /Users/apple/Documents/Ki_2_nam_3/Cơ sở ATTT/finalBTL/ATBM/results/figures/saved_nodes_by_k.png


In [8]:
plot_metric_by_dataset(
    summary,
    x_col="k",
    metric_prefix="time_seconds",
    fixed_filters={"seed_num": 10},
    title="Thời gian chạy khi thay đổi k, cố định |S| = 10",
    xlabel="k - số node được chặn",
    ylabel="Thời gian chạy trung bình (giây)",
    filename="time_seconds_by_k.png",
    yscale="log",
);

Đã lưu: /Users/apple/Documents/Ki_2_nam_3/Cơ sở ATTT/finalBTL/ATBM/results/figures/time_seconds_by_k.png


## 6. Kịch bản 2: thay đổi `|S|`, cố định `k = 100`

In [9]:
plot_metric_by_dataset(
    summary,
    x_col="seed_num",
    metric_prefix="saved_nodes",
    fixed_filters={"k": 100},
    title="Số nút cứu được khi thay đổi |S|, cố định k = 100",
    xlabel="|S| - số seed tin đồn",
    ylabel="Số nút cứu được trung bình",
    filename="saved_nodes_by_seed_num.png",
);

Đã lưu: /Users/apple/Documents/Ki_2_nam_3/Cơ sở ATTT/finalBTL/ATBM/results/figures/saved_nodes_by_seed_num.png


In [10]:
plot_metric_by_dataset(
    summary,
    x_col="seed_num",
    metric_prefix="time_seconds",
    fixed_filters={"k": 100},
    title="Thời gian chạy khi thay đổi |S|, cố định k = 100",
    xlabel="|S| - số seed tin đồn",
    ylabel="Thời gian chạy trung bình (giây)",
    filename="time_seconds_by_seed_num.png",
    yscale="log",
);

Đã lưu: /Users/apple/Documents/Ki_2_nam_3/Cơ sở ATTT/finalBTL/ATBM/results/figures/time_seconds_by_seed_num.png


## 7. Tỷ lệ xấp xỉ `approx_ratio`

Trong kết quả hiện tại, `approx_ratio` thường chỉ có ở biến thể `UB`.

In [11]:
approx_rows = [r for r in summary if not math.isnan(r["approx_ratio_mean"])]
approx_rows[:20]

[{'dataset': 'com-dblp.ungraph',
  'method': 'SandIMIN-UB',
  'variant': 'UB',
  'k': 100,
  'seed_num': 10,
  'runs': 3,
  'saved_nodes_mean': 183.75933333333333,
  'saved_nodes_std': 6.303819820817639,
  'time_seconds_mean': 3628.53,
  'time_seconds_std': 1194.5169065777177,
  'approx_ratio_mean': 0.20405733333333334,
  'approx_ratio_std': 0.011371780174332137},
 {'dataset': 'com-dblp.ungraph',
  'method': 'SandIMIN-UB',
  'variant': 'UB',
  'k': 100,
  'seed_num': 20,
  'runs': 3,
  'saved_nodes_mean': 256.64033333333333,
  'saved_nodes_std': 3.2343485176049502,
  'time_seconds_mean': 4234.143333333333,
  'time_seconds_std': 1438.0185515609085,
  'approx_ratio_mean': 0.22754966666666668,
  'approx_ratio_std': 0.006118699562270843},
 {'dataset': 'com-dblp.ungraph',
  'method': 'SandIMIN-UB',
  'variant': 'UB',
  'k': 100,
  'seed_num': 30,
  'runs': 2,
  'saved_nodes_mean': 296.1585,
  'saved_nodes_std': 4.994295195520577,
  'time_seconds_mean': 1823.48,
  'time_seconds_std': 235.975

In [12]:
plot_metric_by_dataset(
    summary,
    x_col="k",
    metric_prefix="approx_ratio",
    fixed_filters={"seed_num": 10},
    title="Approximation ratio khi thay đổi k, cố định |S| = 10",
    xlabel="k - số node được chặn",
    ylabel="Approximation ratio trung bình",
    filename="approx_ratio_by_k.png",
);

Đã lưu: /Users/apple/Documents/Ki_2_nam_3/Cơ sở ATTT/finalBTL/ATBM/results/figures/approx_ratio_by_k.png


## 8. Xuất bảng tổng hợp CSV

In [13]:
def write_pivot_csv(rows, fixed_filters, x_col, value_col, out_path):
    data = filter_rows(rows, fixed_filters)
    datasets = sorted({r["dataset"] for r in data})
    methods = ordered_methods(data)
    xs = sorted({r[x_col] for r in data})
    lookup = {(r["dataset"], r["method"], r[x_col]): r[value_col] for r in data}

    with out_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["dataset", "method"] + xs)
        for dataset in datasets:
            for method in methods:
                if not any((r["dataset"] == dataset and r["method"] == method) for r in data):
                    continue
                values = []
                for x in xs:
                    value = lookup.get((dataset, method, x), math.nan)
                    values.append("" if math.isnan(value) else round(value, 6))
                writer.writerow([dataset, method] + values)


write_pivot_csv(summary, {"seed_num": 10}, "k", "saved_nodes_mean", RESULTS_DIR / "plot_table_k_saved.csv")
write_pivot_csv(summary, {"seed_num": 10}, "k", "time_seconds_mean", RESULTS_DIR / "plot_table_k_time.csv")
write_pivot_csv(summary, {"k": 100}, "seed_num", "saved_nodes_mean", RESULTS_DIR / "plot_table_s_saved.csv")
write_pivot_csv(summary, {"k": 100}, "seed_num", "time_seconds_mean", RESULTS_DIR / "plot_table_s_time.csv")

print("Đã lưu hình vào:", FIGURES_DIR)
print("Đã lưu bảng CSV vào:", RESULTS_DIR)

Đã lưu hình vào: /Users/apple/Documents/Ki_2_nam_3/Cơ sở ATTT/finalBTL/ATBM/results/figures
Đã lưu bảng CSV vào: /Users/apple/Documents/Ki_2_nam_3/Cơ sở ATTT/finalBTL/ATBM/results
